In [1]:
import glob
import pandas as pd

paths = sorted(glob.glob('../data/raw/cellvoltages_*.csv'))
print(paths)
N_CELLS = 84

['../data/raw\\cellvoltages_2026-03-03-21-21-48.csv', '../data/raw\\cellvoltages_2026-03-11-15-48-37.csv', '../data/raw\\cellvoltages_2026-03-27-18-51-29.csv', '../data/raw\\cellvoltages_2026-03-27-19-01-44.csv', '../data/raw\\cellvoltages_2026-03-27-19-03-55.csv', '../data/raw\\cellvoltages_2026-03-27-19-14-37.csv']


In [2]:
sessions = []
for path in paths:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    DROP_COLS = ([f'Cell Resistance {i}' for i in range(1, N_CELLS + 1)] + 
                [f'Open Cell Voltage {i}' for i in range(1, N_CELLS + 1)]) # Impedance Analysis TBA

    CV_COLS = [f'Cell Voltage {i}' for i in range(1, N_CELLS + 1)] # Cell Voltage Columns

    df = df.drop(columns=DROP_COLS)
    df = df.dropna(axis=1, how='all')   # Delete BMS trailing comma
    df.columns = df.columns.str.strip()

    df['Pack Current'] = df['Pack Current'] * -1 # Make positive when Charging (human intuitive)

    df['Time'] = pd.to_datetime( 
        df['Time'].str.replace(r'\s+[A-Z]{2,4}\s+', ' ', regex=True),
        format='%a %b %d %H:%M:%S %Y')

    df['elapsed_s'] = (df['Time'] - df['Time'].iloc[0]).dt.total_seconds()

    df['session_id'] = path.split('cellvoltages_')[1].replace('.csv', '')
    sessions.append(df)

if not sessions:
    raise ValueError("No CSV files loaded. Check file paths.")

collated = pd.concat(sessions, ignore_index=True)
collated.to_parquet('../data/collated.parquet')